# Phase 6 — Autoencoder for Manifold Learning

**Goal:** Train a neural network that learns to compress 3D manifold data into a 2D latent space — and reconstruct it back — purely from the data itself, with no geometric assumptions.

```
Input (3D)  →  Encoder  →  Latent (2D)  →  Decoder  →  Reconstruction (3D)
   x        →  64→32→16  →     z        →  16→32→64  →      x̂
                          ↑
                   This is our embedding
```

**Loss:** MSE between input `x` and reconstruction `x̂`  
**Optimiser:** Adam with step LR decay  
**Kernel:** `manifold-discovery`

In [ ]:
# Cell 1 — Imports & path fix
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd())
while not (ROOT / 'environment.yml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT = {ROOT}  |  src found: {(ROOT / "src").exists()}')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.manifold import trustworthiness

from src.datasets.generators import load_all_datasets
from src.methods.autoencoder import (Autoencoder, train_autoencoder,
                                      encode_dataset, run_all_autoencoders,
                                      get_device)
from src.utils.plotting import plot_2d, savefig
from src.config import CFG

FIGURES  = CFG['figures_dir']
MODELS   = CFG['models_dir']
SEED     = CFG['random_seed']
DEVICE   = get_device()

print(f'PyTorch version : {torch.__version__}')
print(f'Training device : {DEVICE}')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 100})

In [ ]:
# Cell 2 — Load saved datasets
DATA_DIR = CFG['data_dir']
file_map = {
    'Swiss Roll':   'swiss_roll.npz',
    'S-Curve':      's_curve.npz',
    'Torus':        'torus.npz',
    'Mobius Strip': 'mobius_strip.npz',
}

datasets = {}
for name, fname in file_map.items():
    path = DATA_DIR / fname
    if path.exists():
        d = np.load(path)
        datasets[name] = (d['X'], d['t'])
        print(f'  Loaded {name}: X={d["X"].shape}')
    else:
        print(f'  {fname} not found — regenerating')
        datasets = load_all_datasets(n_samples=CFG['n_samples'], noise=0.1, seed=SEED)
        break

In [ ]:
# Cell 3 — Inspect the model architecture before training
model = Autoencoder(input_dim=3, hidden_dims=[64, 32, 16], latent_dim=2)
print(model)
print(f'\nTotal trainable parameters: {model.count_params():,}')
print()

# Sanity check: forward pass with random data
x_test  = torch.randn(8, 3)
z_test  = model.encode(x_test)
x_hat   = model(x_test)
print(f'Input shape      : {x_test.shape}')
print(f'Latent shape     : {z_test.shape}  ← this is our 2D embedding')
print(f'Reconstruction   : {x_hat.shape}')
print(f'\nInitial MSE loss : {torch.nn.MSELoss()(x_hat, x_test).item():.4f}  (random weights)')

In [ ]:
# Cell 4 — Train on Swiss Roll first (fastest feedback)
# Watch the loss drop — this is the network learning the manifold structure

X_sr, t_sr = datasets['Swiss Roll']

print('Training autoencoder on Swiss Roll...')
model_sr, history_sr = train_autoencoder(
    X_sr,
    latent_dim  = 2,
    hidden_dims = [64, 32, 16],
    epochs      = 200,
    lr          = 1e-3,
    batch_size  = 256,
    device      = DEVICE,
    seed        = SEED,
    log_every   = 20,
)

In [ ]:
# Cell 5 — Loss curve: did the network converge?

losses = history_sr['train_loss']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Swiss Roll autoencoder — training loss', fontsize=12)

# Full curve
axes[0].plot(losses, color='#185FA5', linewidth=1.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE loss')
axes[0].set_title('Full training curve'); axes[0].grid(alpha=0.3)

# Log scale — shows convergence detail
axes[1].semilogy(losses, color='#185FA5', linewidth=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE loss (log scale)')
axes[1].set_title('Log scale — convergence detail'); axes[1].grid(alpha=0.3)

plt.tight_layout()
savefig(fig, FIGURES / '05_ae_loss_swiss_roll.png')
plt.show()

print(f'Initial loss : {losses[0]:.6f}')
print(f'Final loss   : {losses[-1]:.6f}')
print(f'Reduction    : {losses[0]/losses[-1]:.1f}x')
print(f'Training time: {history_sr["fit_time_s"]:.1f}s')

In [ ]:
# Cell 6 — Latent space: did the network unroll the Swiss Roll?

Z_sr = encode_dataset(model_sr, X_sr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Autoencoder — Swiss Roll latent space vs 3D original', fontsize=13)

# 3D original
ax3 = fig.add_subplot(121, projection='3d')
ax3.scatter(X_sr[:, 0], X_sr[:, 1], X_sr[:, 2],
            c=t_sr, cmap='viridis', s=6, alpha=0.7)
ax3.set_title('Original 3D', fontsize=11)
ax3.set_xlabel('X'); ax3.set_ylabel('Y'); ax3.set_zlabel('Z')

# 2D latent space
ax2 = fig.add_subplot(122)
sc  = ax2.scatter(Z_sr[:, 0], Z_sr[:, 1], c=t_sr, cmap='viridis', s=6, alpha=0.8)
plt.colorbar(sc, ax=ax2, label='Manifold param t')
ax2.set_title('Autoencoder 2D latent space', fontsize=11)
ax2.set_xticks([]); ax2.set_yticks([])

plt.tight_layout()
savefig(fig, FIGURES / '05_ae_latent_swiss_roll.png')
plt.show()
print('A clean colour gradient = the network learned the manifold structure.')

In [ ]:
# Cell 7 — Train on all four datasets and save checkpoints
# This is the main training run — takes 5-15 minutes total

print('Training autoencoders on all four datasets...')
print('(Each runs for 200 epochs — grab a coffee)')
print()

ae_results = run_all_autoencoders(
    datasets,
    latent_dim  = 2,
    hidden_dims = [64, 32, 16],
    epochs      = 200,
    lr          = 1e-3,
    batch_size  = 256,
    seed        = SEED,
    save_dir    = MODELS,
)

In [ ]:
# Cell 8 — Loss curves for all four datasets

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Autoencoder training loss — all datasets', fontsize=13)

for ax, (name, res) in zip(axes.flat, ae_results.items()):
    losses = res['history']['train_loss']
    ax.semilogy(losses, color='#185FA5', linewidth=1.5)
    t_s    = res['history']['fit_time_s']
    final  = losses[-1]
    ax.set_title(f'{name}  |  final loss={final:.5f}  ({t_s:.0f}s)', fontsize=10)
    ax.set_xlabel('Epoch', fontsize=9)
    ax.set_ylabel('MSE (log)', fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
savefig(fig, FIGURES / '05_ae_loss_all.png')
plt.show()

In [ ]:
# Cell 9 — Latent spaces for all four datasets

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Autoencoder 2D latent spaces — all datasets', fontsize=13)

for ax, (name, (X, t)) in zip(axes.flat, datasets.items()):
    Z  = ae_results[name]['X_emb']
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=t, cmap='viridis', s=5, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='t')
    final_loss = ae_results[name]['history']['train_loss'][-1]
    ax.set_title(f'{name}  loss={final_loss:.5f}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
savefig(fig, FIGURES / '05_ae_latent_all.png')
plt.show()

In [ ]:
# Cell 10 — Reconstruction quality: original vs reconstructed points
# Shows how well the decoder rebuilds 3D from the 2D latent code

fig, axes = plt.subplots(2, 4, figsize=(18, 8),
                          subplot_kw={'projection': '3d'})
fig.suptitle('Original vs Reconstructed 3D — all datasets', fontsize=13)

for col, (name, (X, t)) in enumerate(datasets.items()):
    model = ae_results[name]['model'].eval()
    X_norm = (X - model.X_mean) / model.X_std
    with torch.no_grad():
        X_hat_norm = model(torch.FloatTensor(X_norm)).numpy()
    X_hat = X_hat_norm * model.X_std + model.X_mean

    mse = np.mean((X - X_hat) ** 2)

    # Original
    axes[0, col].scatter(X[:, 0], X[:, 1], X[:, 2],
                          c=t, cmap='viridis', s=4, alpha=0.6)
    axes[0, col].set_title(f'{name}\nOriginal', fontsize=9)
    axes[0, col].tick_params(labelsize=6)

    # Reconstructed
    axes[1, col].scatter(X_hat[:, 0], X_hat[:, 1], X_hat[:, 2],
                          c=t, cmap='viridis', s=4, alpha=0.6)
    axes[1, col].set_title(f'Reconstructed\nMSE={mse:.4f}', fontsize=9)
    axes[1, col].tick_params(labelsize=6)

plt.tight_layout()
savefig(fig, FIGURES / '05_ae_reconstruction.png')
plt.show()

In [ ]:
# Cell 11 — Effect of latent dimension: 2D vs 3D vs 5D bottleneck
# On Swiss Roll only — shows how bottleneck size affects latent quality

latent_dims = [2, 3, 5]
fig, axes   = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Bottleneck size effect on Swiss Roll latent space', fontsize=12)

for ax, ld in zip(axes, latent_dims):
    print(f'Training latent_dim={ld}...')
    m, h = train_autoencoder(X_sr, latent_dim=ld, epochs=150,
                               device=DEVICE, seed=SEED, verbose=False)
    Z    = encode_dataset(m, X_sr)
    # For 3D+ latent: project first 2 dims for visualisation
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=t_sr, cmap='viridis', s=5, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='t')
    final = h['train_loss'][-1]
    ax.set_title(f'latent_dim={ld}  loss={final:.5f}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
savefig(fig, FIGURES / '05_ae_latent_dim_sweep.png')
plt.show()
print('latent_dim=2 forces maximum compression — more structure visible in the 2D plot.')
print('latent_dim=5 has lower loss but the 2D projection shows only 2 of 5 dims.')

In [ ]:
# Cell 12 — Trustworthiness + full summary metrics table

rows = []
for name, (X, t) in datasets.items():
    res    = ae_results[name]
    Z      = res['X_emb']
    losses = res['history']['train_loss']

    # Reconstruction MSE in original space
    model  = res['model'].eval()
    X_norm = (X - model.X_mean) / model.X_std
    with torch.no_grad():
        X_hat_norm = model(torch.FloatTensor(X_norm)).numpy()
    X_hat  = X_hat_norm * model.X_std + model.X_mean
    recon_mse = float(np.mean((X - X_hat) ** 2))

    tw = trustworthiness(X, Z, n_neighbors=10) if np.std(Z) > 1e-6 else None

    rows.append({
        'Dataset':            name,
        'Final train loss':   f'{losses[-1]:.6f}',
        'Reconstruction MSE': f'{recon_mse:.4f}',
        'Trustworthiness':    f'{tw:.3f}' if tw else 'n/a',
        'Fit time (s)':       f'{res["history"]["fit_time_s"]:.1f}',
        'Parameters':         f'{res["model"].count_params():,}',
    })

df = pd.DataFrame(rows).set_index('Dataset')
print('Autoencoder results summary:')
print(df.to_string())

csv_path = ROOT / 'results' / 'metrics' / '05_autoencoder_metrics.csv'
df.to_csv(csv_path)
print(f'\nSaved -> results/metrics/05_autoencoder_metrics.csv')

## Phase 6 Summary

| Finding | Detail |
|---|---|
| Architecture | 3→64→32→16→**2**→16→32→64→3  (total ~8,000 params) |
| Loss function | MSE between input and reconstruction |
| Optimiser | Adam lr=1e-3, StepLR decay at epoch 100 |
| Swiss Roll | Unrolls cleanly — latent space shows smooth gradient |
| S-Curve | Clean arc in latent space |
| Torus | Collapses to ring like all other methods |
| Möbius Strip | Partial structure — non-orientable surface is hard |
| Bottleneck size | latent_dim=2 forces most compression and clearest structure |
| Checkpoints | Saved to `results/models/*.pt` — reload with `torch.load` |

**Key difference from classical methods:** The autoencoder learns a *parametric* mapping — given any new point, you can encode it without re-running the whole algorithm. Isomap and LLE are transductive (the embedding only covers the training set).

**Next:** `06_evaluation.ipynb` — Phase 7, compare all 7 methods with unified metrics.